In [ ]:
import pandas as pd
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import importlib
import numpy as np

from kfirlib.comparisons import *
from kfirlib import pubmed, gpt

In [ ]:
importlib.reload(pubmed)

In [ ]:
DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables\fromJamalah9_6'

## Read etc.

In [ ]:
dfs = []
for file in os.listdir(DIR):
    dfs.append(pd.read_excel(os.path.join(DIR, file)))
dfs.sort(key=len)

In [ ]:
tables = {}
for key, df in zip(['final', '495', 'yanay', 'all'], dfs):
    tables[key] = df

In [ ]:
for key, df in tables.items():
    print(key)
    title_col = 'Title' if 'Title' in df.columns else 'title'
    assert title_col in df.columns, 'No title column'
    print(sum(df[title_col].isna()))
    df.drop(df[df[title_col].isna()].index, inplace=True)
    df['title_extract'] = df[title_col].apply(extract_letters)

    if 'Abstract' in df.columns:
        print(df['Abstract'].isna().sum())
    elif 'abstract' in df.columns:
        print(df['abstract'].isna().sum())
    else:
        print('no abstracts')
        df['abstract'] = ''

In [ ]:
for key, df in tables.items():
    abstract_col = 'Abstract' if 'Abstract' in df.columns else 'abstract'
    ab = df[abstract_col]
    bad_ab = ab.isna() | (ab == '')
    print(f'{key}: {bad_ab.sum()} / {len(df)}')

In [ ]:
counts = {key: {key: np.nan for key in tables.keys()} for key in tables.keys()}
for key1 in tables.keys():
    for key2 in tables.keys():
        if key1 == key2:
            continue

        tables[key1][f'in_{key2}'] = tables[key1]['title_extract'].isin(tables[key2]['title_extract'])
        counts[key1][key2] = tables[key1][f'in_{key2}'].sum()
    counts[key1]['size'] = len(tables[key1])

In [ ]:
count_df = pd.DataFrame(counts).transpose()
count_df

## Add abstracts

In [ ]:
df_to_use = tables['yanay']
df_to_use.columns

In [ ]:
df_to_use = pubmed.add_lines_by_pubmed_id(df_to_use, 'PMID')

## ChatGPT

In [ ]:
SYSTEM_PROMPT = '''
You are a scientist. You are given a set of inclusion and exclusion criteria and a list of articles. You must decide
whether to include each article and explain why. 
Try your best even if the abstract is missing.

The format should be as follows:

```
## ARTICLE 1

<reasoning>

INCLUDE or EXCLUDE
```

You must make a decision for **each** input and write it only once.
'''
CRITERIA = '''
## INCLUSION CRITERIA
Studies that focus on barriers, enablers, or facilitators of vaccination
Geography- global, Conducted in developed countries (mid-to-high income countries)
Peer reviewed articles
Articles published in English
Articles published since 2010
'''

In [ ]:
del df_to_use['abstract_x']
df_to_use.rename(columns={'abstract_y': 'abstract'}, inplace=True)

In [ ]:
df_to_use.rename(columns={'Publication Year': 'Year'}, inplace=True)
df_to_use.columns

In [ ]:
df = gpt.check_eligibility(
    df_to_use, 
    system_prompt=SYSTEM_PROMPT,
    criteria=CRITERIA,
    model=gpt.Models.MINI)
df.decision.mean()

In [ ]:
yanay_eligibility_df = df

## Save

In [ ]:
for key, df in tables.items():
    if key == 'yanay':
        continue
    yanay_eligibility_df[f'in_{key}'] = yanay_eligibility_df['title_extract'].isin(tables[key]['title_extract'])

In [ ]:
yanay_eligibility_df.to_excel(os.path.join(DIR, 'yanay_with_eligibility.xlsx'), index=False)

In [ ]:
for key, df in tables.items():
    if key == 'yanay':
        continue

    df.to_excel(os.path.join(DIR, f'{key}_with_inclusions.xlsx'), index=False)